# 질의셋과 정답 문서 ID

전체 30개 질문을 읽고 dev 15개로 설정을 검토합니다.

## 실행 환경 준비

이 노트북은 제공 PDF에서 실제 텍스트를 추출합니다. 답변 생성 API는 호출하지 않습니다.

### 평가 행을 코드에서 읽는 순서

`json.loads(line)`은 JSONL 한 줄을 사전으로 반환합니다. `gold_doc_ids`는 답 문장이 아니라 근거 문서 ID 목록이며, 검색 청크는 부모 문서 ID로 바꾸고 중복을 제거한 뒤 비교합니다. `split`은 설정 선택용 dev와 마지막 확인용 test를 나누는 라벨입니다. 여러 문서가 정답인 질문은 하나만 찾으면 되는지 모두 필요한지 라벨 지침에서 먼저 정해야 합니다.

In [1]:
from pathlib import Path
import json, math, re
from collections import Counter
from pypdf import PdfReader

# 현재 폴더에서 저장소의 공개 RAG 자료를 찾습니다.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "labs/data/rag-corpus/manifest.json").exists())
corpus = root / "labs/data/rag-corpus"
dataset_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
dataset = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines()]
evaluation_items = [item for item in dataset if item["split"] == "dev"]
print(f"전체 질문: {len(dataset)}개")
print(f"설정 선택용 dev 질문: {len(evaluation_items)}개")

전체 질문: 30개
설정 선택용 dev 질문: 15개


## 공개 PDF를 페이지 단위 후보로 준비

스캔본은 OCR이 필요하므로 이번 기초 검색에서 제외합니다.

In [2]:
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
pages = []
for item in manifest:
    if item["tier"] == "scanned":
        continue
    reader = PdfReader(corpus / item["file"])
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"doc_id": item["doc_id"], "page": page_no, "text": text})
print(f"검색 후보: {len(pages)}개 페이지")

검색 후보: 90개 페이지


## 같은 검색기를 모든 질문에 적용

단어와 한글 두 글자 조각의 겹침을 사용합니다. 점수가 같으면 문서 ID와 쪽 번호로 순서를 고정합니다.

In [3]:
def tokens(text):
    normalized = re.sub(r"\s+", " ", text.lower())
    words = re.findall(r"[가-힣a-z0-9]+", normalized)
    compact = re.sub(r"[^가-힣a-z0-9]", "", normalized)
    return words + [compact[i:i+2] for i in range(max(0, len(compact)-1))]

def search(question, limit=50):
    query_counts = Counter(tokens(question))
    scored = []
    for page in pages:
        page_counts = Counter(tokens(page["text"]))
        overlap = sum(min(count, page_counts[token]) for token, count in query_counts.items())
        score = overlap / math.sqrt(max(1, sum(page_counts.values())))
        scored.append({"doc_id": page["doc_id"], "page": page["page"], "score": score})
    ranked_pages = sorted(scored, key=lambda row: (-row["score"], row["doc_id"], row["page"]))
    unique_docs, seen = [], set()
    for row in ranked_pages:
        if row["doc_id"] not in seen:
            unique_docs.append(row)
            seen.add(row["doc_id"])
    return unique_docs[:limit]

sample = search(dataset[0]["question"], 5)
print([(row["doc_id"], row["page"], round(row["score"], 3)) for row in sample])

[('E01', 1, 0.926), ('M04', 3, 0.225), ('H03', 1, 0.188), ('H06', 5, 0.142), ('E02', 5, 0.132)]


## 라벨 구조 확인

질문과 정답 문서 ID만 공개합니다. 정답 문장은 검색 평가에 필요하지 않습니다.

In [4]:
required = {"qid", "tier", "question", "type", "gold_doc_ids", "answerable", "split"}
assert all(required == set(item) for item in dataset)
assert all("answer" not in item for item in dataset)
for item in dataset[:3]:
    print(item["qid"], item["tier"], item["split"], item["gold_doc_ids"], item["question"])

Q001 easy dev ['E01'] 미르시립도서관의 정기 휴관일은 언제인가요?
Q002 easy dev ['E01'] 도서는 한 번에 몇 권까지, 며칠 동안 빌릴 수 있나요?
Q003 easy dev ['E01'] 책을 3일 늦게 반납하면 어떤 불이익이 있나요?


## 값 하나를 바꿔 비교

설정 선택용 dev 질문 수를 5개와 15개로 바꾸고 문서 분포가 어떻게 달라지는지 확인합니다.

In [5]:
for size in (5, 15):
    subset = evaluation_items[:size]
    counts = Counter(doc_id for item in subset for doc_id in item["gold_doc_ids"])
    print(size, "문항:", dict(sorted(counts.items())))

5 문항: {'E01': 5}
15 문항: {'E01': 5, 'H01': 5, 'M01': 5}
